# KTH LRCN Temporal Clipping Replica

This notebook is a replica and explanatory version of the KTH temporal clipping workflow. It shows how frames are sampled from a video, how the sequence is extracted over time, and why this is useful for an LRCN model.

The main idea is to keep only a few representative frames from each action video so that the model learns motion patterns instead of processing every frame in the clip.

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected.")

## Step 1: Define the temporal sampling strategy

We choose a fixed number of frames and a fixed step between them. In this example, we select 16 frames with a gap of 6 frames between each selected sample.

This creates a temporal clip that represents the motion pattern of the action without using all video frames.

In [ ]:
num_frames = 16
frame_step = 6
sample_indices = np.arange(
    0,
    num_frames * frame_step,
    frame_step
)

print("Sample indices:", sample_indices)

## Step 2: Open the video and read selected frames

The KTH dataset contains action videos such as walking, running, handclapping, and handwaving. We select a video from the walking class and read only the frames at the sample indices.

In [ ]:
from pathlib import Path

base_dir = Path(r"C:\NNDL Lab Proj\KTH")
dataset_path = base_dir / "Dataset" if (base_dir / "Dataset").exists() else base_dir
video_path = dataset_path / "walking" / "person01_walking_d1_uncomp.avi"
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise FileNotFoundError(f"Video not found: {video_path}")

frames = []

for index in sample_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ret, frame = cap.read()

    if ret:
        frames.append(frame)

cap.release()

print("Number of frames loaded:", len(frames))
print("Frame shape:", frames[0].shape)

## Step 3: Display the sampled frames

The following code visualizes the selected frames in a grid. This helps us confirm that the sampled frames capture the motion progression of the action.

In [ ]:
plt.figure(figsize=(12, 8))

for i, frame in enumerate(frames):
    plt.subplot(4, 4, i + 1)

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.imshow(frame_rgb)
    plt.title(f"Frame {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Explanation of the method

This method is called temporal clipping because it clips the video into a smaller sequence of time-ordered frames.

Why it matters:
- videos are heavy to process
- not every frame is needed to represent an action
- sampled frames still preserve motion information
- the resulting sequence can be used as input to an LRCN model

In a full project, this process would be repeated for all videos across all action classes, and the clips would be paired with class labels before training.

## Final note

This notebook is a simple but important preprocessing step in the KTH LRCN pipeline. It shows how action videos are converted from raw video frames into a manageable sequence of motion samples that can be used for learning.